# 02 — Synthetic Data Generation

## Objective

Generate a small, fully synthetic, reproducible dataset for a fictional bank ("Aurora Trust Bank") that every later notebook in this project reuses: an unstructured **documents** dataset (for parsing, classification, extraction, chunking, embeddings, RAG) and a structured **transactions** dataset (for the SQL tool notebook).

This notebook only *creates* data. It does not parse, chunk, embed, or retrieve anything -- that starts in Group B.

## What We Will Learn

- How to design a small synthetic dataset that is realistic enough to be useful, without any real customer data
- The difference between **unstructured** data (document text) and **structured** data (transaction records) and why this project needs both
- How to make data generation reproducible (fixed content, fixed random seed)
- How to persist the same dataset two ways: as a Delta table (for SQL/DataFrame work) and as plain files in a Unity Catalog Volume (for file-based ingestion in Notebook 03)

## Prerequisites

- Completed `01_databricks_genai_environment.ipynb` (same widget pattern is reused here)
- `CREATE SCHEMA` and `CREATE VOLUME` privileges on your target Unity Catalog catalog
- A cluster or SQL warehouse attached to this notebook

## Conceptual Explanation

**Why synthetic data, and why now.** Every notebook from here on needs *something* to parse, chunk, embed, retrieve, or query. Building that "something" once, here, keeps every later notebook focused on its own concept instead of re-inventing sample data. All content below is invented for this project -- "Aurora Trust Bank" does not exist.

**Unstructured vs. structured.** The *documents* dataset is unstructured: each row is mostly free text (a policy, a procedure, a product description) with a little metadata attached. The *transactions* dataset is structured: every row has the same fixed fields and types, suited to aggregation ("how many wire transfers last month?") rather than reading. Group B and C notebooks work on documents; the SQL tools notebook (13) works on transactions.

**Reproducibility.** The 15 documents below are hand-written, not randomly generated -- their content, category, and date are fixed in code, so re-running this notebook always produces the same documents. The transactions dataset *is* randomly generated (we need volume, not hand-crafted realism, for aggregation examples), so we fix `random.seed(...)` -- re-running with the same seed reproduces the same rows.

**Two storage forms, one dataset.** Later notebooks need this data in two different shapes:
- As a **Delta table**, queryable with SQL/DataFrames -- useful for classification, extraction, and the SQL tool notebook.
- As **plain files in a Unity Catalog Volume** (`.txt` per document, `.csv` for transactions) -- useful for Notebook 03, which is specifically about ingesting *files* (paths, volumes, binary vs. structured), not tables.

We write both so Notebook 03 has real files to read, without having to regenerate anything.

## Example Data

**Documents (15 rows, 5 categories x 3 each):** Product, Operations, Compliance, Customer Service, Technical -- matching the categories used later in Notebook 05 (`ai_classify`).

**Transactions (~250 rows):** fictional account activity across 20 accounts and 6 branches, with a transaction type, channel, status, and amount -- enough variety to support aggregation questions later (e.g. "total wire transfers by branch").

## Implementation

### Step 1 — Parameterize catalog, schema, and volume

Same pattern as Notebook 01: adjust the defaults if `main` / `genai_lab` aren't available to you. The volume is new here -- it's where the file-based copies of this data will live.

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema (created if missing)")
dbutils.widgets.text("volume_name", "synthetic_data", "Volume (created if missing)")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")

documents_table = f"{catalog_name}.{schema_name}.documents"
transactions_table = f"{catalog_name}.{schema_name}.transactions"
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

print(f"Documents table:    {documents_table}")
print(f"Transactions table: {transactions_table}")
print(f"Volume path:        {volume_path}")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
dbutils.fs.mkdirs(f"{volume_path}/documents")
dbutils.fs.mkdirs(f"{volume_path}/transactions")

### Step 2 — Define the 15 synthetic documents

Hand-written, fixed content -- 3 documents in each of 5 categories, with a fictional department and a fixed `created_date` so the dataset is identical on every run.

In [ ]:
# All content below is entirely fictional (Aurora Trust Bank does not exist).
sample_documents = [
    (1,  "Checking Account Overview", "Product", "Retail Banking", "2025-01-15",
     "Aurora Trust Bank's Everyday Checking account is a fee-free checking product for personal "
     "customers. It includes a debit card, mobile check deposit, and no minimum balance requirement. "
     "Overdraft protection is optional and must be enrolled separately. Interest is not paid on "
     "balances in this account tier."),
    (2,  "Fixed Term Deposit Product Guide", "Product", "Retail Banking", "2025-02-10",
     "The Aurora Fixed Term Deposit offers a guaranteed interest rate for terms of 6, 12, 24, or 36 "
     "months. Early withdrawal before maturity incurs a penalty equal to 90 days of accrued interest. "
     "Interest compounds monthly and is credited to the linked checking account. Minimum opening "
     "deposit is 1,000 fictional currency units."),
    (3,  "Personal Line of Credit Terms", "Product", "Retail Banking", "2025-03-05",
     "The Aurora Personal Line of Credit provides revolving access to funds up to an approved limit. "
     "Interest accrues only on the drawn balance, calculated daily. Customers can repay and redraw "
     "funds without reapplying. Annual review of the credit limit is performed based on account "
     "conduct."),
    (4,  "Wire Transfer Procedure", "Operations", "Payments Operations", "2025-01-20",
     "Outbound wire transfers must be verified against the sender's registered signature and a "
     "callback confirmation for amounts above 10,000 units. Transfers are processed in two daily "
     "batches, at 11:00 and 15:00. Same-day cutoff for the afternoon batch is 14:30. Any transfer "
     "flagged by the sanctions screening system is held for manual review before release."),
    (5,  "Cash Deposit Handling Procedure", "Operations", "Branch Operations", "2025-02-18",
     "Cash deposits above 5,000 units require dual counting by two tellers before the transaction is "
     "posted. All cash drawers are reconciled at the end of each shift, not only at end of day. "
     "Discrepancies over 50 units must be logged in the branch incident register and reported to the "
     "branch manager the same day."),
    (6,  "End of Day Reconciliation Procedure", "Operations", "Branch Operations", "2025-03-22",
     "Each branch reconciles teller cash positions, ATM cash levels, and the day's transaction log "
     "before close of business. Any unresolved discrepancy is escalated to Regional Operations within "
     "one business day. The reconciliation report is retained for seven years to satisfy the record "
     "retention policy."),
    (7,  "KYC Policy Summary", "Compliance", "Compliance", "2025-01-08",
     "All new customers must complete identity verification using a government-issued photo ID and "
     "proof of address dated within 90 days. Enhanced due diligence applies to politically exposed "
     "persons and high-risk jurisdictions. Customer risk ratings are reviewed at account opening and "
     "re-assessed annually or upon material change in activity."),
    (8,  "Anti-Money Laundering Monitoring Policy", "Compliance", "Compliance", "2025-02-25",
     "Transaction monitoring rules flag activity inconsistent with a customer's stated profile, "
     "including rapid movement of funds and structuring below reporting thresholds. Flagged "
     "transactions are reviewed by an AML analyst within two business days. Confirmed suspicious "
     "activity is reported through the internal escalation process and filed with the relevant "
     "authority where required."),
    (9,  "Data Retention and Privacy Policy", "Compliance", "Compliance", "2025-04-01",
     "Customer records are retained for a minimum of seven years after account closure, in line with "
     "regulatory requirements. Personal data is only shared with third parties with explicit customer "
     "consent or a legal obligation. Customers may request a copy of their held data or its deletion, "
     "subject to regulatory retention limits."),
    (10, "Disputing a Card Charge", "Customer Service", "Contact Center", "2025-01-30",
     "Customers can dispute an unrecognized card charge within 60 days of the statement date. The "
     "contact center opens a case, issues a provisional credit within 10 business days where eligible, "
     "and requests supporting documentation from the merchant's acquiring bank. Final resolution is "
     "communicated to the customer within 45 days."),
    (11, "Lost or Stolen Card Procedure", "Customer Service", "Contact Center", "2025-03-12",
     "A lost or stolen card is blocked immediately upon customer report through any channel. A "
     "replacement card is issued within 5-7 business days, or same-day at a branch for urgent cases. "
     "Any transactions after the reported loss time are treated as unauthorized pending "
     "investigation."),
    (12, "Updating Contact Information", "Customer Service", "Contact Center", "2025-04-15",
     "Customers can update phone number and email through online banking without additional "
     "verification. Updating a mailing address or legal name requires identity re-verification, "
     "either in branch or through the secure document upload flow. Changes are confirmed by "
     "notification to both the old and new contact details."),
    (13, "Core Banking API Overview", "Technical", "Engineering", "2025-01-05",
     "The Core Banking API exposes account, transaction, and customer resources over REST, returning "
     "JSON. All endpoints require an OAuth2 bearer token scoped to the calling application. Rate "
     "limits are enforced per API key, with current limits returned in response headers. A sandbox "
     "environment mirrors production schemas with synthetic data only."),
    (14, "Authentication and API Keys Guide", "Technical", "Engineering", "2025-02-14",
     "API keys are issued per integration and must be rotated at least every 90 days. Each key is "
     "scoped to specific resource permissions rather than granted blanket access. Requests without a "
     "valid bearer token are rejected with a 401 response. Key usage is logged for audit and anomaly "
     "detection purposes."),
    (15, "Webhook Event Notifications Reference", "Technical", "Engineering", "2025-03-28",
     "Webhook subscribers receive events for transaction posted, account status changed, and card "
     "blocked, delivered as signed JSON payloads. Delivery is retried with exponential backoff for up "
     "to 24 hours on failure. Subscribers should verify the payload signature before processing. "
     "Duplicate delivery is possible and consumers should handle events idempotently."),
]

document_columns = ["doc_id", "title", "category", "department", "created_date", "content"]
documents_df = spark.createDataFrame(sample_documents, document_columns)
display(documents_df)

### Step 3 — Save documents as a Delta table, and as individual `.txt` files

The Delta table is for SQL/DataFrame use later. The `.txt` files exist specifically so Notebook 03 has real files, at a real Volume path, to practice file-based ingestion on.

In [ ]:
documents_df.write.mode("overwrite").saveAsTable(documents_table)
print(f"Wrote {documents_df.count()} rows to {documents_table}")

In [ ]:
for row in documents_df.collect():
    slug = row["title"].lower().replace(" ", "_").replace("/", "_")
    file_path = f"{volume_path}/documents/{row['doc_id']:02d}_{slug}.txt"
    file_text = (
        f"Title: {row['title']}\n"
        f"Category: {row['category']}\n"
        f"Department: {row['department']}\n"
        f"Created: {row['created_date']}\n\n"
        f"{row['content']}\n"
    )
    dbutils.fs.put(file_path, file_text, overwrite=True)

print(f"Wrote {documents_df.count()} .txt files to {volume_path}/documents")
display(dbutils.fs.ls(f"{volume_path}/documents"))

### Step 4 — Generate the structured transactions dataset

`random.seed(42)` makes this reproducible: every run generates the exact same ~250 rows. Amounts and dates vary by transaction type so aggregations later (e.g. "average wire transfer amount") produce sensible results rather than uniform noise.

In [ ]:
import random
from datetime import date, timedelta

random.seed(42)

REFERENCE_DATE = date(2025, 4, 30)  # fixed "as of" date, so date math stays reproducible
ACCOUNTS = [f"ACC-{1000 + i}" for i in range(20)]
BRANCHES = [f"BR-{i:03d}" for i in range(1, 7)]

# (transaction_type, channel choices, amount range, relative frequency weight)
TRANSACTION_TYPES = [
    ("Deposit",       ["Branch", "Mobile", "ATM"],            (20, 3000),  30),
    ("Withdrawal",     ["Branch", "ATM"],                       (20, 1000),  25),
    ("Wire Transfer",  ["Branch", "Online", "Wire Desk"],       (100, 15000), 10),
    ("Card Payment",   ["Online", "Mobile"],                     (5, 500),    30),
    ("Fee",            ["Online", "Mobile"],                     (2, 35),      5),
]
STATUS_WEIGHTS = [("Completed", 90), ("Pending", 7), ("Failed", 3)]


def weighted_choice(options_with_weights):
    options = [o for o, _ in options_with_weights]
    weights = [w for _, w in options_with_weights]
    return random.choices(options, weights=weights, k=1)[0]


transaction_rows = []
for i in range(1, 251):
    txn_type, channels, (low, high), weight = random.choices(
        TRANSACTION_TYPES, weights=[w for *_, w in TRANSACTION_TYPES], k=1
    )[0]
    txn_date = REFERENCE_DATE - timedelta(days=random.randint(0, 89))
    transaction_rows.append((
        f"TXN-{i:06d}",
        random.choice(ACCOUNTS),
        txn_date.isoformat(),
        round(random.uniform(low, high), 2),
        txn_type,
        random.choice(channels),
        weighted_choice(STATUS_WEIGHTS),
        random.choice(BRANCHES),
    ))

transaction_columns = [
    "transaction_id", "account_id", "transaction_date", "amount",
    "transaction_type", "channel", "status", "branch_code",
]
transactions_df = spark.createDataFrame(transaction_rows, transaction_columns)
display(transactions_df.limit(10))

### Step 5 — Save transactions as a Delta table, and as a single CSV file

In [ ]:
transactions_df.write.mode("overwrite").saveAsTable(transactions_table)
print(f"Wrote {transactions_df.count()} rows to {transactions_table}")

csv_text = transactions_df.toPandas().to_csv(index=False)
dbutils.fs.put(f"{volume_path}/transactions/transactions.csv", csv_text, overwrite=True)
print(f"Wrote transactions.csv to {volume_path}/transactions")

## Inspect the Output

- Confirm document counts per category (should be exactly 3 each) and eyeball a few `content` values for realism.
- Confirm the transaction count and spot-check that `status` is `Completed` roughly 90% of the time.
- Browse `/Volumes/<catalog>/<schema>/<volume>/documents` in the Catalog Explorer UI and open one `.txt` file to see exactly what Notebook 03 will read.
- Open `transactions.csv` in the same UI (or `dbutils.fs.head(...)`) and check the header row matches `transaction_columns`.

In [ ]:
%sql
-- Update the catalog/schema below if you changed the widgets in Step 1
SELECT category, COUNT(*) AS doc_count
FROM main.genai_lab.documents
GROUP BY category
ORDER BY category

In [ ]:
%sql
SELECT status, COUNT(*) AS txn_count, ROUND(AVG(amount), 2) AS avg_amount
FROM main.genai_lab.transactions
GROUP BY status
ORDER BY txn_count DESC

## Experimentation Section

1. Change `random.seed(42)` to a different number and re-run Step 4 -- confirm the transaction rows change, then change it back and confirm you get the *original* 250 rows again. This is what "reproducible" means in practice.
2. Add a 16th document in a new category (e.g. "Fraud") and re-run Steps 2-3. Does the `.txt` file loop pick it up automatically? Why?
3. Increase the transaction count from 250 to 2,500 and compare how long Step 4/5 take -- this is a preview of why later notebooks care about pipeline performance.
4. Change `STATUS_WEIGHTS` so `Failed` is 30% instead of 3%, and re-run the last SQL cell -- watch `avg_amount` and counts shift.
5. Open one of the generated `.txt` files directly in the Catalog Explorer file preview and compare it with the same row in the `documents` Delta table -- same content, two different shapes.

## Common Errors / Limitations

- **`CREATE VOLUME` permission denied** -- Volumes are a Unity Catalog object like schemas/tables; you need `CREATE VOLUME` on the schema. Ask an admin if this fails, or use a catalog/schema where you already have it.
- **`dbutils.fs.put` overwrite behavior** -- `overwrite=True` silently replaces existing files. That's intentional here (so re-running this notebook is idempotent), but be aware of it if you reuse this pattern elsewhere.
- **This is not realistic-scale data** -- 15 documents and 250 transactions are sized for *inspectability*, not for testing performance or scale. Do not draw performance conclusions from this dataset.
- **`%sql` cells have hardcoded paths again** -- as in Notebook 01, update `main.genai_lab...` in the SQL cells by hand if you changed the widget defaults.

## Summary

You now have a reusable, reproducible synthetic dataset in two shapes: `documents` and `transactions` Delta tables, plus `.txt` and `.csv` files under a Unity Catalog Volume. Every notebook in Groups B-F builds on top of exactly this data -- no notebook from here on needs to invent its own sample content.

## Suggested Exercises

- Write a short SQL query joining `transactions` to `documents` on a shared fictional attribute you add yourself (e.g. tag each branch with a related "Branch Operations" document).
- Compute, per account, the number of transactions and total amount -- this is exactly the shape of query Notebook 13 (SQL tools) will wrap as a callable tool.
- When you're ready, move on to **`03_document_ingestion_basics.ipynb`**, which reads the `.txt` files you just wrote back in as raw documents.